# Activity 1: RAGAS Evaluation with Cost Analysis

We compare two RAG stacks answering questions over the same feline-health PDF (the 2021 AAHA/AAFP Feline Life Stage Guidelines):

- **Open-source stack (Fireworks):** `gpt-oss-20b` for generation + `qwen3-embedding-8b` for retrieval.
- **Closed stack (OpenAI):** `gpt-4.1-mini` for generation + `text-embedding-3-small` for retrieval.

Both use the identical prompt, chunking, and retrieval settings, so the only thing that changes is the models. We score both with **RAGAS** on three things the assignment asks for:

| Assignment dimension | RAGAS metric(s) |
|---|---|
| Retrieval quality | `LLMContextPrecisionWithReference`, `LLMContextRecall` |
| Answer faithfulness | `Faithfulness` |
| End-to-end accuracy | `FactualCorrectness`, plus `ResponseRelevancy` as support |

A single **OpenAI `gpt-4.1-mini` judge** scores both stacks (same judge = fair comparison). Every model call is traced to **LangSmith**, and we also capture token usage directly from each response to compute cost per query and cost at scale.

In [1]:
# ragas 0.4.3 hard-imports langchain_community.chat_models.vertexai, which was
# removed in langchain-community 0.4. We do not use Vertex AI, so we register a
# tiny stub module and ragas imports cleanly against the newer LangChain stack.
import sys, types
_stub = types.ModuleType("langchain_community.chat_models.vertexai")
_stub.ChatVertexAI = type("ChatVertexAI", (), {})
sys.modules["langchain_community.chat_models.vertexai"] = _stub

import os, time
import pandas as pd
from dotenv import load_dotenv, find_dotenv

# usecwd=True so it finds the project .env from the notebook's working directory.
load_dotenv(find_dotenv(usecwd=True))

# Route every LLM/judge call to a LangSmith project for token + cost tracing.
os.environ.setdefault("LANGSMITH_TRACING", "true")
os.environ["LANGSMITH_PROJECT"] = "session10-ragas-eval"

for key in ("FIREWORKS_API_KEY", "OPENAI_API_KEY", "LANGSMITH_API_KEY"):
    print(f"{key}: {'set' if os.environ.get(key) else 'MISSING'}")

FIREWORKS_API_KEY: set
OPENAI_API_KEY: set
LANGSMITH_API_KEY: set


## Pricing

Prices are USD per 1M tokens, pulled from each provider's public pricing (July 2026). Embeddings bill on input tokens only.

- Fireworks `gpt-oss-20b`: $0.07 in / $0.30 out ([Fireworks pricing](https://fireworks.ai/pricing))
- Fireworks `qwen3-embedding-8b`: $0.20 in ([Fireworks pricing](https://fireworks.ai/pricing))
- OpenAI `gpt-4.1-mini`: $0.40 in / $1.60 out ([OpenAI pricing](https://openai.com/api/pricing/))
- OpenAI `text-embedding-3-small`: $0.02 in ([OpenAI pricing](https://openai.com/api/pricing/))

In [2]:
# USD per 1M tokens.
PRICING = {
    "Fireworks (open-source)": {
        "chat_in": 0.07, "chat_out": 0.30, "embed_in": 0.20,   # gpt-oss-20b + qwen3-embedding-8b
    },
    "OpenAI (gpt-4.1-mini)": {
        "chat_in": 0.40, "chat_out": 1.60, "embed_in": 0.02,   # gpt-4.1-mini + text-embedding-3-small
    },
}

## Load and chunk the PDF

Same loader and token-aware splitter the RAG app uses, so both stacks index identical chunks.

In [3]:
import tiktoken
from langchain_community.document_loaders import DirectoryLoader, PyMuPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

_enc = tiktoken.encoding_for_model("gpt-4o")
def tk_len(text: str) -> int:
    return len(_enc.encode(text))

documents = DirectoryLoader("data", glob="**/*.pdf", loader_cls=PyMuPDFLoader).load()
chunks = RecursiveCharacterTextSplitter(
    chunk_size=750, chunk_overlap=0, length_function=tk_len
).split_documents(documents)

corpus_tokens = sum(tk_len(c.page_content) for c in chunks)
print(f"{len(documents)} pages -> {len(chunks)} chunks, {corpus_tokens:,} tokens to embed once")

/var/folders/mm/g297wvhn773bc64j8l__p83h0000gn/T/ipykernel_54006/2948377842.py:2: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import DirectoryLoader, PyMuPDFLoader


22 pages -> 42 chunks, 24,212 tokens to embed once


## Build the two RAG stacks

`build_rag` wires a retriever + a generation model behind the same context-only prompt. Each call returns the retrieved contexts, the answer, and the exact token counts (read from the response, so it works for both providers).

In [4]:
from langchain_openai import ChatOpenAI
from langchain_openai.embeddings import OpenAIEmbeddings
from langchain_qdrant import QdrantVectorStore
from langchain_core.prompts import ChatPromptTemplate

FW_BASE = "https://api.fireworks.ai/inference/v1"
FW_KEY = os.environ["FIREWORKS_API_KEY"]
OA_KEY = os.environ["OPENAI_API_KEY"]

RAG_PROMPT = ChatPromptTemplate.from_messages([(
    "human",
    "\n#CONTEXT:\n{context}\n\nQUERY:\n{query}\n\n"
    "Use the provided context to answer the query. Only use the provided context. "
    'If the answer is not in the context, respond with "I don\'t know".',
)])

def build_rag(chat, embeddings, name):
    """Return a callable question -> {contexts, response, token counts} for one stack."""
    store = QdrantVectorStore.from_documents(
        chunks, embedding=embeddings, location=":memory:", collection_name=f"eval_{name}"
    )
    retriever = store.as_retriever(search_kwargs={"k": 4})

    def run(question: str) -> dict:
        ctx_docs = retriever.invoke(question)
        context = "\n\n".join(d.page_content for d in ctx_docs)
        msg = (RAG_PROMPT | chat).invoke({"query": question, "context": context})
        usage = getattr(msg, "usage_metadata", None) or {}
        return {
            "retrieved_contexts": [d.page_content for d in ctx_docs],
            "response": msg.content,
            "chat_in": usage.get("input_tokens", 0),
            "chat_out": usage.get("output_tokens", 0),
            "embed_in": tk_len(question),  # query embedding tokens (retrieval)
        }
    return run

fireworks_rag = build_rag(
    ChatOpenAI(model="accounts/fireworks/models/gpt-oss-20b", temperature=0,
               openai_api_key=FW_KEY, openai_api_base=FW_BASE),
    OpenAIEmbeddings(model="accounts/fireworks/models/qwen3-embedding-8b",
                     openai_api_key=FW_KEY, openai_api_base=FW_BASE,
                     check_embedding_ctx_length=False, dimensions=4096),
    "fireworks",
)
openai_rag = build_rag(
    ChatOpenAI(model="gpt-4.1-mini", temperature=0, openai_api_key=OA_KEY),
    OpenAIEmbeddings(model="text-embedding-3-small", openai_api_key=OA_KEY),
    "openai",
)
print("both RAG stacks ready")

both RAG stacks ready


## Test set

A small curated set of questions with reference answers taken straight from the guidelines PDF. Curated (not synthetic) keeps the ground truth trustworthy and gives both stacks the identical questions.

In [5]:
TESTSET = [
    {"question": "According to the 2021 AAHA/AAFP guidelines, what are the feline life stages and their age ranges?",
     "reference": "Five stages: kitten (birth up to 1 year), young adult (1 through 6 years), mature adult (7 to 10 years), senior (over 10 years), and an end-of-life stage."},
    {"question": "At what age is a cat considered a senior, and how often should senior cats be examined?",
     "reference": "A cat is senior once it is over 10 years old. Senior cats should be seen at least every 6 months, while the minimum for adult cats is an annual examination."},
    {"question": "Which core vaccines do the guidelines list for cats?",
     "reference": "The core vaccines are rabies virus, feline herpesvirus type 1 (FHV-1), feline calicivirus (FCV), and feline panleukopenia virus (FPV). Non-core vaccines are added based on exposure and susceptibility."},
    {"question": "What does the parasite control recommendation say cats should be protected against?",
     "reference": "Year-round broad-spectrum parasite control with products effective against heartworms, intestinal parasites, and fleas."},
    {"question": "What three measurements are recommended to monitor a cat's weight and physical condition?",
     "reference": "Body weight, body condition score (BCS), and muscle condition score (MCS) should be evaluated and recorded."},
]
print(f"{len(TESTSET)} evaluation questions")

5 evaluation questions


## Run both stacks and capture tokens

Same questions through each stack. We keep the retrieved contexts and answers (for RAGAS) and the token counts (for cost).

In [6]:
def run_pipeline(rag, label):
    rows = []
    for i, item in enumerate(TESTSET, 1):
        t = time.time()
        out = rag(item["question"])
        rows.append({"user_input": item["question"], "reference": item["reference"], **out})
        print(f"  [{label}] q{i}  {time.time()-t:4.1f}s  in={out['chat_in']:>4}  out={out['chat_out']:>4}")
    return rows

print("Fireworks stack:")
fw_rows = run_pipeline(fireworks_rag, "fw")
print("OpenAI stack:")
oa_rows = run_pipeline(openai_rag, "oa")

pd.set_option("display.max_colwidth", 90)
preview = pd.DataFrame([
    {"question": fw["user_input"][:55], "fireworks_answer": fw["response"][:70], "openai_answer": oa["response"][:70]}
    for fw, oa in zip(fw_rows, oa_rows)
])
preview

Fireworks stack:


  [fw] q1   5.9s  in=1363  out= 149


  [fw] q2  79.1s  in=2598  out= 203


  [fw] q3   4.3s  in=2586  out= 235


  [fw] q4  75.7s  in=3017  out= 741


...

  [oa] q2   1.9s  in=2705  out=  35


  [oa] q3   1.2s  in=2519  out=  46


  [oa] q4   2.3s  in=2825  out= 120


  [oa] q5   1.3s  in=2802  out=  50


,question,fireworks_answer,openai_answer
0,"According to the 2021 AAHA/AAFP guidelines, what are th",I don't know.,"According to the 2021 AAHA/AAFP Feline Life Stage Guidelines, the cat’"
1,"At what age is a cat considered a senior, and how often",A cat is considered a **senior** when it is **older than 10 years**.,A cat is considered a senior at 10 years of age and older. Senior cats
2,Which core vaccines do the guidelines list for cats?,The guidelines list the following core vaccines for cats:\n\n- Rabies vi,The guidelines list the following core vaccines for cats: rabies virus
3,What does the parasite control recommendation say cats,The parasite‑control recommendation states that cats should be protect,The parasite control recommendation says cats should be protected agai
4,What three measurements are recommended to monitor a ca,The recommended trio of measurements to track a cat’s weight and physi,The three measurements recommended to monitor a cat's weight and physi


## Score with RAGAS

One OpenAI judge scores both stacks on the five metrics. Scores run 0 to 1 (higher is better).

In [7]:
from ragas import evaluate, EvaluationDataset, SingleTurnSample
from ragas.llms import LangchainLLMWrapper
from ragas.embeddings import LangchainEmbeddingsWrapper
from ragas.metrics import (
    Faithfulness, ResponseRelevancy,
    LLMContextPrecisionWithReference, LLMContextRecall, FactualCorrectness,
)

judge_llm = LangchainLLMWrapper(ChatOpenAI(model="gpt-4.1-mini", temperature=0, openai_api_key=OA_KEY))
judge_emb = LangchainEmbeddingsWrapper(OpenAIEmbeddings(model="text-embedding-3-small", openai_api_key=OA_KEY))

metrics = [
    LLMContextPrecisionWithReference(llm=judge_llm),  # retrieval quality
    LLMContextRecall(llm=judge_llm),                  # retrieval quality
    Faithfulness(llm=judge_llm),                      # answer faithfulness
    ResponseRelevancy(llm=judge_llm, embeddings=judge_emb),  # answer addresses the question
    FactualCorrectness(llm=judge_llm),                # end-to-end accuracy vs reference
]

def to_dataset(rows):
    return EvaluationDataset(samples=[SingleTurnSample(
        user_input=r["user_input"], retrieved_contexts=r["retrieved_contexts"],
        response=r["response"], reference=r["reference"]) for r in rows])

fw_result = evaluate(dataset=to_dataset(fw_rows), metrics=metrics)
oa_result = evaluate(dataset=to_dataset(oa_rows), metrics=metrics)

fw_scores = fw_result.to_pandas()
oa_scores = oa_result.to_pandas()
metric_cols = [c for c in fw_scores.columns
               if c not in ("user_input", "retrieved_contexts", "response", "reference")]

quality = pd.DataFrame({
    "Fireworks (open-source)": fw_scores[metric_cols].mean(numeric_only=True),
    "OpenAI (gpt-4.1-mini)": oa_scores[metric_cols].mean(numeric_only=True),
}).round(3)
quality["winner"] = quality.apply(
    lambda r: "OpenAI" if r["OpenAI (gpt-4.1-mini)"] > r["Fireworks (open-source)"]
    else ("Fireworks" if r["Fireworks (open-source)"] > r["OpenAI (gpt-4.1-mini)"] else "tie"), axis=1)
print("Average RAGAS scores (0-1, higher is better)")
quality

/var/folders/mm/g297wvhn773bc64j8l__p83h0000gn/T/ipykernel_54006/3311142390.py:4: DeprecationWarning: Importing Faithfulness from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import Faithfulness
  from ragas.metrics import (
/var/folders/mm/g297wvhn773bc64j8l__p83h0000gn/T/ipykernel_54006/3311142390.py:4: DeprecationWarning: Importing ResponseRelevancy from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import ResponseRelevancy
  from ragas.metrics import (
/var/folders/mm/g297wvhn773bc64j8l__p83h0000gn/T/ipykernel_54006/3311142390.py:4: DeprecationWarning: Importing LLMContextPrecisionWithReference from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import LLMContextPrecisionWithReference

Evaluating:   0%|          | 0/25 [00:00<?, ?it/s]

LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.


LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.


LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.


...

LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.


LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.


LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.


Average RAGAS scores (0-1, higher is better)


,Fireworks (open-source),OpenAI (gpt-4.1-mini),winner
llm_context_precision_with_reference,0.717,0.917,OpenAI
context_recall,1.000,1.000,tie
faithfulness,0.800,1.000,OpenAI
answer_relevancy,0.740,0.762,OpenAI
factual_correctness(mode=f1),0.490,0.544,OpenAI


## Cost analysis

Cost per query from the captured token counts, then projected to scale. The one-time cost to embed the whole PDF is listed separately since it is paid once at index time, not per query.

In [8]:
def cost_breakdown(rows, provider):
    p = PRICING[provider]
    chat_in = sum(r["chat_in"] for r in rows)
    chat_out = sum(r["chat_out"] for r in rows)
    embed_in = sum(r["embed_in"] for r in rows)
    total = (chat_in * p["chat_in"] + chat_out * p["chat_out"] + embed_in * p["embed_in"]) / 1e6
    n = len(rows)
    return {
        "avg chat in (tok)": round(chat_in / n),
        "avg chat out (tok)": round(chat_out / n),
        "cost / query ($)": total / n,
        "cost / 1K queries ($)": total / n * 1_000,
        "cost / 1M queries ($)": total / n * 1_000_000,
        "one-time index ($)": corpus_tokens * p["embed_in"] / 1e6,
    }

cost = pd.DataFrame({
    "Fireworks (open-source)": cost_breakdown(fw_rows, "Fireworks (open-source)"),
    "OpenAI (gpt-4.1-mini)": cost_breakdown(oa_rows, "OpenAI (gpt-4.1-mini)"),
})
fw_q = cost.loc["cost / query ($)", "Fireworks (open-source)"]
oa_q = cost.loc["cost / query ($)", "OpenAI (gpt-4.1-mini)"]
money_rows = {"cost / query ($)", "cost / 1K queries ($)", "cost / 1M queries ($)", "one-time index ($)"}

def _fmt(row, v):
    if row in money_rows:
        return f"${v:,.4f}" if v < 1 else f"${v:,.2f}"
    return f"{int(v):,}"

# Build an all-string table from scratch (avoids pandas 3.0 dtype-assignment errors).
cost_fmt = pd.DataFrame({col: {row: _fmt(row, cost.loc[row, col]) for row in cost.index}
                         for col in cost.columns})
print(f"OpenAI costs {oa_q / fw_q:.1f}x more per query than the open-source stack")
cost_fmt

OpenAI costs 4.0x more per query than the open-source stack


,Fireworks (open-source),OpenAI (gpt-4.1-mini)
avg chat in (tok),"2,479","2,514"
avg chat out (tok),347,76
cost / query ($),$0.0003,$0.0011
cost / 1K queries ($),$0.2808,$1.13
cost / 1M queries ($),$280.82,"$1,128.32"
one-time index ($),$0.0048,$0.0005


## Results and takeaways

Quality: on this test set the OpenAI gpt-4.1-mini stack scored higher on every metric or tied. It matched the open-source stack on context recall (1.00 vs 1.00) and beat it on context precision (0.92 vs 0.72), faithfulness (1.00 vs 0.80), answer relevancy (0.76 vs 0.74), and factual correctness (0.54 vs 0.49). The open-source gpt-oss-20b stack was more conservative and answered "I don't know" on the life-stages question when its retrieval missed the exact chunk, which keeps it honest but costs it on accuracy.

Cost: the open-source Fireworks stack was much cheaper, about $0.0003 per query versus $0.0011 for OpenAI, so the closed model costs roughly four times more per query. At scale that is about $281 versus $1,128 per million queries. The one-time cost to embed the whole PDF was under a cent for both.

The decision: this is the quality versus cost tradeoff the session is about. The closed model answers are a bit more accurate, but the open-source stack is four times cheaper and can also be run privately, which matters for sensitive data. For a high-volume or privacy-sensitive app the open-source stack is a reasonable choice even at slightly lower accuracy. For a low-volume app where every answer has to be exactly right, paying more per query for the closed model can be worth it.

LangSmith: every model call in this notebook is traced under the session10-ragas-eval project, where the token usage and latency per call line up with the cost numbers above.